# F1 Improvements Targeting Postings Without a Company Profile

The error analysis in `interpretation.ipynb` showed that most mistakes of the selected model occur on postings with an empty company profile: in repeat 0 of the development CV they contain 151 of the 179 missed fake postings and 83 of the 90 false alarms. This notebook tests three changes aimed at that group. The baseline is the selected Step 2 SVM (`STEP2_CONFIG`); all runs use the original folds (seed 0), so they are paired with the saved Step 2 results. The test set is not used.

### Candidates and decision rule (fixed before running)

| # | Candidate | Change |
|---|---|---|
| A | **Group thresholds** | Same model and scores, but one decision threshold for postings with an empty profile and one for the rest, chosen jointly on the inner out-of-fold scores to maximise overall F1. |
| B | **Empty-profile interaction** | A second word TF-IDF block (vocabulary from empty-profile postings) that is non-zero only for postings with an empty profile, plus the empty-profile flag. The SVM can learn separate word weights for these postings. |
| C | **SVM + logistic regression ensemble** | Average of the standardised scores of the Step 2 SVM and a logistic regression on the same features (`C = 10`, class-weighted; fixed, not tuned). |

**Decision rule.** The target here is the F1 at the chosen threshold, so the rule is applied to F1: a candidate is adopted when its mean paired F1 gain over the baseline is positive **and** it is better in at least 12 of the 15 folds. PR-AUC is reported alongside.

**Combination.** If two or more candidates pass, the passing changes are applied together once and kept only if the combination also passes the rule against the best single candidate. No other variants are added after seeing the results.

In [1]:
import sklearn

if sklearn.__version__ != "1.6.1":
    raise RuntimeError(
        f"This notebook uses scikit-learn 1.6.1 (found {sklearn.__version__}). "
        "Install the pinned versions with: pip install -r requirements.txt"
    )

import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from fakejob.data import load_split
from fakejob.evaluation import compare, passes_rule, repeated_group_cv
from fakejob.features import STEP2_CONFIG, build_pipeline, posting_features
from fakejob.models import ScoreAverage

pd.set_option("display.precision", 3)
RESULTS_DIR = Path("results")
METRICS = ["average_precision", "roc_auc", "f1", "precision", "recall"]
GROUP = "empty_company_profile"

split = load_split()
features = posting_features(split.model_df)
dev_index = split.dev_index
X_dev = features.loc[dev_index]
y_dev = split.y.loc[dev_index]
groups_dev = split.groups.loc[dev_index]
print(f"Development data: {len(dev_index)} postings, {int(y_dev.sum())} fake")
pd.crosstab(X_dev[GROUP].map({0: "profile filled", 1: "profile empty"}), y_dev.map({0: "real", 1: "fake"}))

Development data: 15001 postings, 724 fake


fraudulent,fake,real
empty_company_profile,,
profile empty,484,2314
profile filled,240,11963


## Helper

Every run also records the group-threshold metrics (`f1_grouped`, ...), so candidate A and its combinations with B or C need no extra training. `as_grouped()` turns a result into one whose threshold metrics are the group-threshold ones, so the same paired comparison can be used.

In [2]:
results = {}

def run(name, estimator):
    start = time.time()
    result = repeated_group_cv(
        estimator, X_dev, y_dev, groups_dev, name=name,
        n_splits=5, n_repeats=3, inner_splits=3, seed=0, n_jobs=4,
        threshold_group_column=GROUP,
    )
    results[name] = result
    result.folds.to_csv(RESULTS_DIR / f"f1_{name}_folds.csv", index=False)
    print(f"{name}: {time.time() - start:.0f}s")
    return result

def as_grouped(result, name):
    folds = result.folds.copy()
    for metric in ["f1", "precision", "recall"]:
        folds[metric] = folds[f"{metric}_grouped"]
    return replace(result, name=name, folds=folds)

def judge(reference, candidates):
    rows = []
    for cand in candidates:
        comparison = compare(reference, cand, metrics=METRICS)
        f1, ap = comparison.loc["f1"], comparison.loc["average_precision"]
        rows.append({
            "variant": cand.name,
            "F1": f"{cand.folds['f1'].mean():.3f} ± {cand.folds['f1'].std():.3f}",
            "Δ F1": f1["mean_diff"],
            "F1 wins": f1["folds_improved"],
            "Δ precision": comparison.loc["precision", "mean_diff"],
            "Δ recall": comparison.loc["recall", "mean_diff"],
            "PR-AUC": f"{cand.folds['average_precision'].mean():.3f}",
            "Δ PR-AUC": ap["mean_diff"],
            "passes F1 rule": passes_rule(comparison, metric="f1"),
        })
    return pd.DataFrame(rows).set_index("variant")

## Baseline and Candidate A: Group Thresholds

The assertion checks that the baseline reproduces the saved Step 2 fold results exactly.

In [3]:
baseline = run("baseline", build_pipeline(STEP2_CONFIG))
saved_step2 = pd.read_csv(RESULTS_DIR / "step2_combined_folds.csv")
assert np.allclose(baseline.folds[METRICS].values, saved_step2[METRICS].values)

group_thresholds = as_grouped(baseline, "A_group_thresholds")
table_a = judge(baseline, [group_thresholds])
display(table_a)

threshold_columns = [f"threshold_{GROUP}_0", f"threshold_{GROUP}_1"]
print("Thresholds (mean ± std over folds): single",
      f"{baseline.folds['threshold'].mean():+.3f} ± {baseline.folds['threshold'].std():.3f};",
      "filled profile", f"{baseline.folds[threshold_columns[0]].mean():+.3f} ± {baseline.folds[threshold_columns[0]].std():.3f};",
      "empty profile", f"{baseline.folds[threshold_columns[1]].mean():+.3f} ± {baseline.folds[threshold_columns[1]].std():.3f}")

baseline: 702s


,F1,Δ F1,F1 wins,Δ precision,Δ recall,PR-AUC,Δ PR-AUC,passes F1 rule
variant,,,,,,,,
A_group_thresholds,0.781 ± 0.063,-0.009,3/15,-0.003,-0.013,0.871,0.0,False


Thresholds (mean ± std over folds): single -0.272 ± 0.070; filled profile -0.285 ± 0.101; empty profile -0.259 ± 0.069


## Candidate B: Empty-Profile Interaction

In [4]:
interaction = run("B_interaction", build_pipeline(STEP2_CONFIG.but(profile_interaction=True)))
table_b = judge(baseline, [interaction])
table_b

B_interaction: 841s


,F1,Δ F1,F1 wins,Δ precision,Δ recall,PR-AUC,Δ PR-AUC,passes F1 rule
variant,,,,,,,,
B_interaction,0.787 ± 0.057,-0.003,5/15,-0.016,0.012,0.875,0.005,False


## Candidate C: SVM + Logistic Regression Ensemble

In [5]:
LOGREG_CONFIG = STEP2_CONFIG.but(classifier="logreg", C=10.0)
ensemble = run("C_ensemble", ScoreAverage([build_pipeline(STEP2_CONFIG), build_pipeline(LOGREG_CONFIG)]))
table_c = judge(baseline, [ensemble])
table_c

C_ensemble: 2076s


,F1,Δ F1,F1 wins,Δ precision,Δ recall,PR-AUC,Δ PR-AUC,passes F1 rule
variant,,,,,,,,
C_ensemble,0.784 ± 0.063,-0.006,6/15,0.003,-0.01,0.867,-0.003,False


## Combination (only if two or more candidates pass)

Combinations with A need no new run: the group-threshold metrics of the B or C run are used. A combination of B and C (ensemble whose members use the interaction features) needs one more run.

In [6]:
passed = {name: table["passes F1 rule"].iloc[0]
          for name, table in [("A", table_a), ("B", table_b), ("C", table_c)]}
print("Passed:", [n for n, ok in passed.items() if ok] or "none")
single = {"A": group_thresholds, "B": interaction, "C": ensemble}

if sum(passed.values()) >= 2:
    if passed["B"] and passed["C"]:
        model = run("BC_ensemble_interaction", ScoreAverage([
            build_pipeline(STEP2_CONFIG.but(profile_interaction=True)),
            build_pipeline(LOGREG_CONFIG.but(profile_interaction=True)),
        ]))
    else:
        model = interaction if passed["B"] else ensemble
    name = "+".join(n for n in "ABC" if passed[n])
    combined = as_grouped(model, name) if passed["A"] else model
    combined = replace(combined, name=name)
    best_single = max((single[n] for n in "ABC" if passed[n]), key=lambda r: r.folds["f1"].mean())
    display(judge(baseline, [combined]))
    vs_best = judge(best_single, [combined])
    print(f"Combination vs best single candidate ({best_single.name}):")
    display(vs_best)
    selected = combined if vs_best["passes F1 rule"].iloc[0] else best_single
elif sum(passed.values()) == 1:
    selected = single[next(n for n, ok in passed.items() if ok)]
else:
    selected = baseline
print("Selected:", selected.name)

Passed: none
Selected: baseline


## Selected Configuration vs Baseline

Results for the two profile groups are shown separately using the out-of-fold scores of repeat 0 (every development posting is scored once), with each fold's own thresholds.

In [7]:
display(compare(baseline, selected, metrics=METRICS))

def by_profile(result, grouped):
    oof = result.oof_scores[result.oof_scores["repeat"] == 0]
    folds = result.folds[result.folds["repeat"] == 0].set_index("fold")
    rows = []
    for fold, part in oof.groupby("fold"):
        empty = X_dev.loc[part.index, GROUP].to_numpy()
        if grouped:
            threshold = np.where(empty == 1, folds.loc[fold, threshold_columns[1]], folds.loc[fold, threshold_columns[0]])
        else:
            threshold = folds.loc[fold, "threshold"]
        rows.append(pd.DataFrame({"empty": empty, "y": y_dev.loc[part.index].to_numpy(),
                                  "flag": (part["score"].to_numpy() >= threshold).astype(int)}))
    data = pd.concat(rows)
    out = {}
    for label, part in data.groupby(data["empty"].map({0: "filled", 1: "empty"})):
        fake, real = part["y"] == 1, part["y"] == 0
        out[label] = {"fake": int(fake.sum()), "detected": int((fake & (part.flag == 1)).sum()),
                      "missed": int((fake & (part.flag == 0)).sum()),
                      "false alarms": int((real & (part.flag == 1)).sum())}
    return pd.DataFrame(out).T

selected_is_grouped = "A" in selected.name.split("_")[0].split("+")
display(pd.concat({"baseline": by_profile(baseline, grouped=False),
                   selected.name: by_profile(selected, grouped=selected_is_grouped)}))

,baseline,candidate,mean_diff,std_diff,folds_improved
metric,,,,,
average_precision,0.871,0.871,0.0,0.0,0/15
roc_auc,0.984,0.984,0.0,0.0,0/15
f1,0.789,0.789,0.0,0.0,0/15
precision,0.834,0.834,0.0,0.0,0/15
recall,0.752,0.752,0.0,0.0,0/15


fake  detected  missed  false alarms
baseline empty    484       333     151            83
         filled   240       212      28             7

## Overview

In [8]:
candidates = [group_thresholds, interaction, ensemble] + [results[n] for n in results if n.startswith("BC_")]
overview = judge(baseline, candidates)
overview.to_csv(RESULTS_DIR / "f1_overview.csv")
overview

,F1,Δ F1,F1 wins,Δ precision,Δ recall,PR-AUC,Δ PR-AUC,passes F1 rule
variant,,,,,,,,
A_group_thresholds,0.781 ± 0.063,-0.009,3/15,-0.003,-0.013,0.871,0.000,False
B_interaction,0.787 ± 0.057,-0.003,5/15,-0.016,0.012,0.875,0.005,False
C_ensemble,0.784 ± 0.063,-0.006,6/15,0.003,-0.010,0.867,-0.003,False


## Findings

**No candidate passes the F1 rule, so the selected Step 2 SVM stays the final model.** All three changes lower the mean F1 slightly and win in at most 6 of the 15 folds.

| Candidate | F1 | Δ F1 | F1 wins | Δ PR-AUC |
|---|---|---|---|---|
| Baseline (Step 2 SVM) | 0.789 | | | |
| A: group thresholds | 0.781 | −0.009 | 3/15 | 0.000 |
| B: empty-profile interaction | 0.787 | −0.003 | 5/15 | +0.005 |
| C: SVM + logistic regression | 0.784 | −0.006 | 6/15 | −0.003 |

- **A: group thresholds.** The two thresholds chosen on the inner folds are close to the single one (on average −0.285 for filled profiles and −0.259 for empty profiles, against −0.272). Splitting the inner data in two makes each threshold noisier, which costs more than the small difference between the groups gains. The scores already rank the two groups on a common scale.
- **B: empty-profile interaction.** Recall rises by 0.012 but precision falls by 0.016, so F1 does not improve. Its PR-AUC gain is small but consistent (+0.005, better in 12 of 15 folds), so it would have passed the PR-AUC rule used in the earlier steps. It is not adopted, because the criterion for this notebook was fixed in advance as F1 and changing it after seeing the results would be selection on the evaluation folds. With group thresholds on top, B reaches F1 0.789, equal to the baseline (6/15 wins).
- **C: ensemble.** The logistic regression is slightly weaker than the SVM on the same features, and averaging their scores does not help (PR-AUC 0.867, F1 0.784). It also takes about three times as long to train.

**Conclusion.** The errors on postings with an empty company profile are not fixed by a separate threshold, separate word weights or a second linear model; these postings carry less information, and the remaining errors look like a limit of the bag-of-words features rather than of the decision rule. The final model is therefore `STEP2_CONFIG` with Platt calibration (development CV: PR-AUC 0.871, F1 0.789, precision 0.834, recall 0.752). The test set has still not been used.